# Build leakage-safe features

This notebook turns the raw **master panel** (one row per delivery) into a modelling table, step by step. It mirrors `features/build_features.py`: the functions below are copied from that script, so both produce identical output. Use the script in the pipeline and this notebook to read and inspect the logic.

**Input:** `data/raw/<scenario>/master_panel.parquet`
**Output:** `data/processed/<scenario>/features.parquet` and `feature_columns.json`

> All data is fully simulated. Columns starting with `true_` are the simulator's hidden truth. They are used here **only** to check that the features behave sensibly, never as model inputs.

## The leakage rule

The dashboard forecasts deliveries **up to 72 hours ahead**. A feature is only valid if it would have been available at that moment. So every history feature for a delivery is computed *as of* a cutoff:

```
cutoff = planned_delivery_datetime − 72 hours
```

and only uses shipments whose outcome was **known** by then, meaning the truck had actually arrived (`actual_delivery_datetime ≤ cutoff`). Receipts that were never scanned never become known, so they never enter any history.

```
 past shipments known ──────────┤ cutoff ├───── 72 h ─────┤ planned arrival
       (usable)                                (forecast horizon)
```

## Setup

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW_DIR = ROOT / "data" / "raw"
OUT_DIR = ROOT / "data" / "processed"

SCENARIO = "drift"   # "clusters", "no_clusters" or "drift"


In [2]:
HORIZON = pd.Timedelta(hours=72)


SUPPLIER_WINDOWS = {"30d": pd.Timedelta(days=30), "90d": pd.Timedelta(days=90)}


REGION_WINDOW = pd.Timedelta(days=7)


PART_SHORTAGE_WINDOW = pd.Timedelta(days=30)


COLD_START_MIN_SHIPMENTS = 20


WINTER_MONTHS = (12, 1, 2)

## Load the master panel

One row per part on each truck. A quick look at size and missing values (the simulator deliberately leaves some gaps).

In [3]:
panel = pd.read_parquet(RAW_DIR / SCENARIO / "master_panel.parquet")
print(f"{len(panel):,} delivery rows, {panel['shipment_id'].nunique():,} shipments, "
      f"{panel['supplier_code'].nunique()} suppliers")
missing = panel.isna().sum()
missing[missing > 0].rename("missing_rows").to_frame()

645,927 delivery rows, 373,198 shipments, 366 suppliers


,missing_rows
actual_delivery_datetime,3265
delay_hours,3265
is_late,3265
distance_km,28726
safety_inv_boxes,16804
planned_usage_boxes,1901
true_drift_month,432947


## Step 1 · Shipment-level outcomes

All parts on one truck share the same delay. Supplier history is therefore counted in **shipments**, not part rows. Otherwise a supplier that ships many parts per truck would count each late truck several times.

This step also adds the `cutoff` for each shipment and the time its outcome became `known`.

In [4]:
def shipment_outcomes(panel: pd.DataFrame) -> pd.DataFrame:
    """One row per truck. Parts on the same truck share one delay, so supplier
    history is counted in shipments, not part rows (otherwise suppliers with
    many parts would be over-weighted)."""
    cols = ["shipment_id", "supplier_code", "plant", "region", "production_date",
            "planned_delivery_datetime", "actual_delivery_datetime", "delay_hours", "is_late"]
    ships = panel[cols].drop_duplicates("shipment_id").copy()
    ships["cutoff"] = ships["planned_delivery_datetime"] - HORIZON
    ships["known_time"] = ships["actual_delivery_datetime"]          # outcome known on arrival
    ships["late"] = ships["is_late"].astype("float")                 # NaN if unscanned
    ships["log_delay"] = np.where(ships["late"] == 1, np.log(ships["delay_hours"].clip(lower=1e-3)), np.nan)
    return ships

In [5]:
ships = shipment_outcomes(panel)
ships[["shipment_id", "supplier_code", "planned_delivery_datetime", "cutoff",
       "known_time", "delay_hours", "late"]].head()

,shipment_id,supplier_code,planned_delivery_datetime,cutoff,known_time,delay_hours,late
0,SH0061953,S0069,2025-01-06 07:00:00,2025-01-03 07:00:00,2025-01-06 08:02:00,1.03,1.0
1,SH0087801,S0091,2025-01-06 07:00:00,2025-01-03 07:00:00,2025-01-06 06:29:00,0.00,0.0
3,SH0135145,S0132,2025-01-06 07:00:00,2025-01-03 07:00:00,2025-01-06 06:28:00,0.00,0.0
4,SH0180285,S0183,2025-01-06 07:00:00,2025-01-03 07:00:00,2025-01-06 06:49:00,0.00,0.0
5,SH0191807,S0195,2025-01-06 07:00:00,2025-01-03 07:00:00,2025-01-06 06:34:00,0.00,0.0


## Step 2 · As-of history with cumulative sums

Recomputing each supplier's history separately for 370k shipments would be slow. Instead:

1. Sort known outcomes by `known_time` and keep a **running total** per supplier (shipments, late shipments, total delay, …).
2. For each shipment, look up the running total at its `cutoff` with `pd.merge_asof`. That gives "everything known so far."
3. For a 30-day window, look up the total at `cutoff − 30 days` too and **subtract**.

For example, if a supplier's running late count is 14 at the cutoff and 9 thirty days earlier, it had 5 late shipments in the window.

In [6]:
def _cumulative_history(events: pd.DataFrame, by: list[str], value_cols: list[str]) -> pd.DataFrame:
    """Running totals per group, ordered by the time each outcome became known."""
    known = events.dropna(subset=["known_time"]).sort_values("known_time").copy()
    known["n"] = 1.0
    sums = ["n", *value_cols]
    known[[f"cum_{c}" for c in sums]] = known.groupby(by)[sums].cumsum()
    return known[[*by, "known_time", *[f"cum_{c}" for c in sums]]]


def _asof(queries: pd.DataFrame, history: pd.DataFrame, by: list[str], time_col: str) -> pd.DataFrame:
    """Cumulative totals at `time_col` for each query row (0 if no history yet)."""
    q = queries[[*by, time_col]].reset_index().sort_values(time_col)
    out = pd.merge_asof(q, history, left_on=time_col, right_on="known_time", by=by,
                        direction="backward", allow_exact_matches=True)
    cum_cols = [c for c in history.columns if c.startswith("cum_")]
    out[cum_cols] = out[cum_cols].fillna(0.0)
    return out.set_index("index").loc[queries.index, cum_cols]


def _window(queries, history, by, time_col, window=None):
    """Totals over (time - window, time]; whole history if window is None."""
    at_t = _asof(queries, history, by, time_col)
    if window is None:
        return at_t
    q = queries[[*by]].copy()
    q["_start"] = queries[time_col] - window
    at_start = _asof(q, history, by, "_start")
    return at_t - at_start.values


def _safe_div(a, b):
    return np.where(b > 0, a / np.where(b > 0, b, 1), np.nan)

### Supplier history features

For each window (all history, 90 days, 30 days):

- **Volume:** `sup_n_ship_*`, how much evidence there is
- **How often late:** `sup_late_rate_*`
- **Expected delay:** `sup_mean_delay_*`, including on-time trucks as zero
- **When late, how late and how variable:** `sup_mean_late_delay_*` and `sup_log_delay_sd_*`. These separate *consistently late* suppliers from *high volatility* ones.

Plus `sup_late_rate_trend` (30-day minus 90-day rate, positive means getting worse), `sup_days_since_late`, `sup_history_days` and a cold-start flag for suppliers with fewer than 20 known shipments.

In [7]:
def supplier_history_features(ships: pd.DataFrame) -> pd.DataFrame:
    ev = ships.assign(
        delay=ships["delay_hours"],
        late_delay=np.where(ships["late"] == 1, ships["delay_hours"], 0.0),
        log_d=ships["log_delay"].fillna(0.0),
        log_d2=ships["log_delay"].fillna(0.0) ** 2,
    )
    hist = _cumulative_history(ev, ["supplier_code"], ["late", "delay", "late_delay", "log_d", "log_d2"])
    feats = pd.DataFrame(index=ships.index)

    for label, window in [("all", None), *SUPPLIER_WINDOWS.items()]:
        w = _window(ships, hist, ["supplier_code"], "cutoff", window)
        n, k = w["cum_n"].to_numpy(), w["cum_late"].to_numpy()
        feats[f"sup_n_ship_{label}"] = n
        feats[f"sup_late_rate_{label}"] = _safe_div(k, n)
        feats[f"sup_mean_delay_{label}"] = _safe_div(w["cum_delay"].to_numpy(), n)
        if label in ("all", "90d"):
            mean_log = _safe_div(w["cum_log_d"].to_numpy(), k)
            var_log = _safe_div(w["cum_log_d2"].to_numpy(), k) - mean_log ** 2
            feats[f"sup_mean_late_delay_{label}"] = _safe_div(w["cum_late_delay"].to_numpy(), k)
            feats[f"sup_log_delay_sd_{label}"] = np.where(k >= 2, np.sqrt(np.clip(var_log, 0, None)), np.nan)

    feats["sup_late_rate_trend"] = feats["sup_late_rate_30d"] - feats["sup_late_rate_90d"]

    # Days since last late shipment / since first known shipment
    known = ships.dropna(subset=["known_time"]).sort_values("known_time")
    last_late = known.assign(t=known["known_time"].where(known["late"] == 1)) \
                     .assign(t=lambda d: d.groupby("supplier_code")["t"].ffill()) \
                     .dropna(subset=["t"])[["supplier_code", "known_time", "t"]]
    first = known.groupby("supplier_code")["known_time"].min().rename("first_known")

    q = ships[["supplier_code", "cutoff"]].reset_index().sort_values("cutoff")
    m = pd.merge_asof(q, last_late, left_on="cutoff", right_on="known_time", by="supplier_code",
                      direction="backward").set_index("index").loc[ships.index]
    feats["sup_days_since_late"] = (ships["cutoff"] - m["t"]).dt.total_seconds() / 86400
    hist_days = (ships["cutoff"] - ships["supplier_code"].map(first)).dt.total_seconds() / 86400
    feats["sup_history_days"] = hist_days.where(hist_days >= 0)
    feats["sup_is_cold_start"] = feats["sup_n_ship_all"] < COLD_START_MIN_SHIPMENTS
    return feats

In [8]:
sup_feats = supplier_history_features(ships)
sup_feats.describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
sup_n_ship_all,373198.0,691.262,580.804,0.000,245.000,535.000,972.000,2928.000
sup_late_rate_all,368478.0,0.134,0.123,0.000,0.036,0.083,0.214,1.000
sup_mean_delay_all,368478.0,1.025,1.512,0.000,0.081,0.489,1.084,19.088
sup_mean_late_delay_all,362352.0,5.528,4.316,0.550,2.306,3.493,8.094,49.605
sup_log_delay_sd_all,355492.0,0.629,0.273,0.004,0.420,0.536,0.833,2.376
sup_n_ship_30d,373198.0,114.624,65.900,0.000,64.000,103.000,165.000,275.000
sup_late_rate_30d,368478.0,0.139,0.132,0.000,0.033,0.086,0.233,1.000
sup_mean_delay_30d,368478.0,1.141,1.765,0.000,0.074,0.449,1.138,19.088
sup_n_ship_90d,373198.0,311.513,196.318,0.000,166.000,262.000,452.000,798.000
sup_late_rate_90d,368478.0,0.137,0.128,0.000,0.034,0.086,0.231,1.000


### Region history

Late rate across all suppliers in a region over the last 7 days. This is meant to pick up regional disruptions such as storms or border delays. Expect it to be a weak signal: most disruptions last 1–4 days, shorter than the 72-hour lag.

In [9]:
def region_history_features(ships: pd.DataFrame) -> pd.DataFrame:
    hist = _cumulative_history(ships, ["region"], ["late"])
    w = _window(ships, hist, ["region"], "cutoff", REGION_WINDOW)
    return pd.DataFrame({"region_late_rate_7d": _safe_div(w["cum_late"].to_numpy(), w["cum_n"].to_numpy())},
                        index=ships.index)

In [10]:
region_feats = region_history_features(ships)
region_feats.describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
region_late_rate_7d,366458.0,0.142,0.05,0.0,0.121,0.135,0.158,1.0


## Step 3 · Features known in advance

Anything on the delivery schedule is known before the truck leaves: the planned hour, weekday, month, winter flag and how many trucks the route runs that day. These need no cutoff.

In [11]:
def calendar_features(ships: pd.DataFrame) -> pd.DataFrame:
    t = ships["planned_delivery_datetime"]
    trips = ships.groupby(["supplier_code", "plant", "production_date"])["shipment_id"].transform("size")
    return pd.DataFrame({
        "planned_hour": t.dt.hour + t.dt.minute / 60,
        "planned_dow": t.dt.dayofweek,
        "planned_month": t.dt.month,
        "is_winter": t.dt.month.isin(WINTER_MONTHS),
        "trips_per_day": trips,
    }, index=ships.index)

In [12]:
cal_feats = calendar_features(ships)
cal_feats.head()

C:\Users\leer1\AppData\Local\Temp\ipykernel_20484\2472658680.py:6: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  "planned_dow": t.dt.dayofweek,


,planned_hour,planned_dow,planned_month,is_winter,trips_per_day
0,7.0,0,1,True,4
1,7.0,0,1,True,4
3,7.0,0,1,True,4
4,7.0,0,1,True,4
5,7.0,0,1,True,4


## Step 4 · Shortage label and shortage-step features

These belong to the **second** step of the pipeline: how much a late truck would hurt.

- `safety_coverage_hours` = safety stock (boxes × pieces per box) ÷ usage per hour, i.e. how long the line can run on safety stock alone.
- `shortage_flag` (a label, not a feature) = the delay was longer than that coverage. It's a simplification that assumes the line runs throughout the delay, so it slightly overstates shortages for trucks that are late overnight.
- `part_shortages_30d` = shortages for this part at this plant in the 30 days before the cutoff, using the same as-of method.

In [13]:
def shortage_features(panel: pd.DataFrame) -> pd.DataFrame:
    df = panel[["part_code", "plant", "planned_delivery_datetime", "actual_delivery_datetime",
                "delay_hours", "safety_inv_boxes", "box_qty", "usage_per_hour"]].copy()
    coverage = df["safety_inv_boxes"].astype("float") * df["box_qty"] / df["usage_per_hour"]
    shortage = (df["delay_hours"] > coverage).astype("float")
    shortage[df["delay_hours"].isna() | coverage.isna()] = np.nan

    events = pd.DataFrame({"part_code": df["part_code"], "plant": df["plant"],
                           "known_time": df["actual_delivery_datetime"],
                           "shortage": shortage.fillna(0.0)})
    hist = _cumulative_history(events, ["part_code", "plant"], ["shortage"])
    q = df[["part_code", "plant"]].assign(cutoff=df["planned_delivery_datetime"] - HORIZON)
    w = _window(q, hist, ["part_code", "plant"], "cutoff", PART_SHORTAGE_WINDOW)

    return pd.DataFrame({
        "safety_coverage_hours": coverage.round(2),
        "part_shortages_30d": w["cum_shortage"].to_numpy(),
        "shortage_flag": shortage.astype("boolean"),
    }, index=panel.index)

## Step 5 · Assemble and label column roles

Join the shipment-level features back to each part row, add the part-level shortage columns and add **missing-value indicators**. Missing values are *not* imputed here, because fill values (e.g. a median) must be learned from the training window only.

Every column gets a role, saved to `feature_columns.json`, so the modelling code can never accidentally use a target, an outcome or a `true_` column as a feature.

In [14]:
COLUMN_ROLES = {
    "id": ["delivery_id", "shipment_id", "supplier_code", "part_code", "plant",
           "production_date", "planned_delivery_datetime", "cutoff"],
    "target": ["delay_hours", "is_late", "shortage_flag"],
    "outcome_do_not_use": ["actual_delivery_datetime"],
    "delay_features": [
        "region", "commodity", "distance_km", "distance_km_missing", "lead_time_days",
        "planned_hour", "planned_dow", "planned_month", "is_winter", "trips_per_day",
        "sup_n_ship_all", "sup_late_rate_all", "sup_mean_delay_all",
        "sup_mean_late_delay_all", "sup_log_delay_sd_all",
        "sup_n_ship_30d", "sup_late_rate_30d", "sup_mean_delay_30d",
        "sup_n_ship_90d", "sup_late_rate_90d", "sup_mean_delay_90d",
        "sup_mean_late_delay_90d", "sup_log_delay_sd_90d", "sup_late_rate_trend",
        "sup_days_since_late", "sup_history_days", "sup_is_cold_start",
        "region_late_rate_7d",
    ],
    "shortage_features": [
        "safety_coverage_hours", "safety_inv_missing", "usage_per_hour", "box_qty",
        "planned_hours", "ordered_qty_boxes", "avail_in_other_plants", "n_plants_using",
        "part_shortages_30d",
    ],
}

In [15]:
ship_feats = pd.concat([ships[["shipment_id", "cutoff"]], sup_feats, region_feats, cal_feats], axis=1)

features = panel.merge(ship_feats, on="shipment_id", how="left", validate="many_to_one")
features = pd.concat([features, shortage_features(features)], axis=1)
features["distance_km_missing"] = features["distance_km"].isna()
features["safety_inv_missing"] = features["safety_inv_boxes"].isna()

truth = [c for c in features.columns if c.startswith("true_")]
ordered = [c for role in COLUMN_ROLES.values() for c in role]
features = features[ordered + truth].sort_values("planned_delivery_datetime").reset_index(drop=True)

print(f"{len(features):,} rows × {features.shape[1]} columns")
pd.Series({role: len(cols) for role, cols in COLUMN_ROLES.items()} | {"evaluation_only": len(truth)},
          name="n_columns").to_frame()

645,927 rows × 60 columns


,n_columns
id,8
target,3
outcome_do_not_use,1
delay_features,28
shortage_features,9
evaluation_only,11


## Step 6 · Leakage check

The fast cumulative-sum method is easy to get subtly wrong, so this test recomputes a random sample of features the slow, obvious way, filtering to shipments known before the cutoff, and asserts they match.

In [16]:
def check_no_leakage(features: pd.DataFrame, panel: pd.DataFrame, n_samples: int = 200, seed: int = 0) -> None:
    """Recompute a sample of supplier features by brute force, keeping only
    shipments known by the cutoff, and confirm they match."""
    ships = shipment_outcomes(panel)
    sample = features.drop_duplicates("shipment_id").sample(n_samples, random_state=seed)
    for _, row in sample.iterrows():
        h = ships[(ships["supplier_code"] == row["supplier_code"]) & (ships["known_time"] <= row["cutoff"])]
        h30 = h[h["known_time"] > row["cutoff"] - SUPPLIER_WINDOWS["30d"]]
        assert len(h) == row["sup_n_ship_all"], "all-history count mismatch"
        assert len(h30) == row["sup_n_ship_30d"], "30-day count mismatch"
        if len(h30):
            assert np.isclose(h30["late"].mean(), row["sup_late_rate_30d"]), "30-day late rate mismatch"
    print(f"  leakage check passed on {n_samples} sampled shipments")

In [17]:
check_no_leakage(features, panel)

  leakage check passed on 200 sampled shipments


## Step 7 · Sanity checks against the hidden truth

This is the one place the `true_` columns are used, to confirm the features carry the signal the simulator put in.

**Do history features track each supplier's true late probability?**

In [18]:
s = features.drop_duplicates("shipment_id")
pd.Series({
    "corr(late_rate_all, true_p_late)": s["sup_late_rate_all"].corr(s["true_p_late"]),
    "corr(late_rate_90d, true_p_late)": s["sup_late_rate_90d"].corr(s["true_p_late"]),
    "shortage rate (all deliveries)": features["shortage_flag"].astype(float).mean(),
    "shortage rate (late deliveries)": features.loc[features["is_late"] == True, "shortage_flag"].astype(float).mean(),
    "cold-start share of shipments": s["sup_is_cold_start"].mean(),
}, name="value").round(3).to_frame()

,value
"corr(late_rate_all, true_p_late)",0.895
"corr(late_rate_90d, true_p_late)",0.744
shortage rate (all deliveries),0.070
shortage rate (late deliveries),0.459
cold-start share of shipments,0.029


**Drift scenario only: do short windows adapt faster?** For suppliers that switch cluster at month 3, compare the error of the 30-day late rate and the all-history late rate against their *current* true late probability. After the switch, the 30-day window should recover within a month while all-history stays wrong. This is the trade-off the drift scenario is designed to test.

In [19]:
if SCENARIO == "drift":
    d = s[s["true_drift_month"] == 3].copy()
    switch = pd.Timestamp("2025-01-06") + pd.DateOffset(months=3)
    current_p = np.where(d["planned_delivery_datetime"] >= switch, d["true_p_late_after"], d["true_p_late"])
    d["error_30d"] = (d["sup_late_rate_30d"] - current_p).abs()
    d["error_all_history"] = (d["sup_late_rate_all"] - current_p).abs()
    display(d.groupby(d["planned_delivery_datetime"].dt.to_period("M"))[["error_30d", "error_all_history"]]
             .mean().round(3).T)
else:
    print("Set SCENARIO = 'drift' to run this check.")

planned_delivery_datetime,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01
error_30d,0.045,0.037,0.031,0.118,0.034,0.030,0.036,0.035,0.032,0.031,0.032,0.033,0.055
error_all_history,0.045,0.033,0.028,0.155,0.142,0.114,0.098,0.087,0.076,0.067,0.060,0.055,0.051


## Step 8 · Save

In [20]:
out = OUT_DIR / SCENARIO
out.mkdir(parents=True, exist_ok=True)
features.to_parquet(out / "features.parquet", index=False)
roles = {**COLUMN_ROLES, "evaluation_only": truth}
(out / "feature_columns.json").write_text(json.dumps(roles, indent=2))
print(f"saved {out / 'features.parquet'}")

saved C:\Users\leer1\Documents\GitHub\bayesian-supplier-risk-engine-jit\data\processed\drift\features.parquet


## Helper for the clustering step

Clusters must be fit **inside each training window**, not on the full year. `supplier_snapshot` summarizes each supplier using only shipments known by a given date. In walk-forward validation you would call it at the end of each training window and fit the clustering on the result.

The example below takes a snapshot at the end of month 6.

In [21]:
def supplier_snapshot(panel: pd.DataFrame, as_of: pd.Timestamp) -> pd.DataFrame:
    """Per-supplier summary using only shipments known by `as_of`.

    Fit clustering on this snapshot at the end of each training window, never
    on the full dataset.
    """
    ships = shipment_outcomes(panel)
    h = ships[ships["known_time"] <= as_of]
    g = h.groupby("supplier_code")
    snap = pd.DataFrame({
        "n_shipments": g.size(),
        "late_rate": g["late"].mean(),
        "mean_delay": g["delay_hours"].mean(),
        "p90_delay": g["delay_hours"].quantile(0.9),
        "mean_late_delay": h[h["late"] == 1].groupby("supplier_code")["delay_hours"].mean(),
        "log_delay_sd": h.groupby("supplier_code")["log_delay"].std(),
    })
    static = panel.drop_duplicates("supplier_code").set_index("supplier_code")[
        ["region", "distance_km", "lead_time_days", "commodity"]]
    return snap.join(static, how="left").reset_index()

In [22]:
as_of = pd.Timestamp("2025-01-06") + pd.DateOffset(months=6)
snapshot = supplier_snapshot(panel, as_of)
print(f"snapshot as of {as_of.date()}: {len(snapshot)} suppliers with known history")
snapshot.head()

snapshot as of 2025-07-06: 356 suppliers with known history


,supplier_code,n_shipments,late_rate,mean_delay,p90_delay,mean_late_delay,log_delay_sd,region,distance_km,lead_time_days,commodity
0,S0001,1326,0.277526,1.371689,3.580,4.942554,0.787241,ON_LOCAL,136.0,1,fasteners
1,S0002,532,0.046992,0.164511,0.000,3.500800,0.712435,ON_REGIONAL,332.0,2,stamping
2,S0003,266,0.033835,0.057068,0.000,1.686667,0.540742,ON_REGIONAL,294.0,1,stamping
3,S0004,398,0.472362,5.649070,14.478,11.959202,0.800053,QC,822.0,2,electrical
4,S0005,398,0.017588,0.071884,0.000,4.087143,0.718161,ON_REGIONAL,162.0,2,interior_trim


## Next steps

- **Walk-forward folds:** split by `planned_delivery_datetime` into expanding training windows, each followed by a test window, and hold out the final period for the last report.
- **Inside each fold:** fit imputation on the training rows only, take a `supplier_snapshot` at the training end, fit the clustering and attach cluster labels to training and test rows.
- **Model inputs:** use only the `delay_features` for the delay models; `shortage_features` feed the conversion to 72-hour shortage probability.